In [3]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import FashionMNIST

In [4]:
from torch.utils.data import DataLoader
from torchvision import transforms

transform  = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5))
])

trainset = FashionMNIST(root = './sample_data', train = True, download = True, transform = transform)
testset = FashionMNIST(root = './sample_data', train = False, download = True, transform = transform)

100%|██████████| 26.4M/26.4M [00:01<00:00, 15.0MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 269kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 5.02MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 13.0MB/s]


In [5]:
trainloader = DataLoader(trainset, batch_size = 64, shuffle = True)
testloader = DataLoader(testset, batch_size = 64)

In [6]:
trainloader

In [7]:
testloader

In [12]:
class CNN(nn.Module):
  def __init__(self):
    super(CNN, self).__init__()

    self.conv_layers = nn.Sequential(
        nn.Conv2d(1, 32, kernel_size = 3, padding = 1),
        nn.ReLU(),
        nn.MaxPool2d(2, 2),

        nn.Conv2d(32, 64, kernel_size = 3, padding = 1),
        nn.ReLU(),
        nn.MaxPool2d(2, 2),

        nn.Conv2d(64, 128, kernel_size = 3, padding =1),
        nn.ReLU(),
        nn.MaxPool2d(2, 2)
    )

    self.fc_layers = nn.Sequential(
        nn.Linear(3*3*128, 256),
        nn.ReLU(),

        nn.Linear(256, 10)
    )

  def forward(self, x):
    x = self.conv_layers(x)
    x = x.view(x.size(0), -1) # flattening
    x = self.fc_layers(x)

    return x



In [13]:
model = CNN()

In [14]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [15]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()

        output = model.forward(images) # FP
        loss = criterion(output, labels) # loss fnx
        loss.backward() # BP
        optimizer.step() # update params

        epoch_training_loss += loss.item()

    print(f"epoch={epoch+1}/{epochs} & loss={epoch_training_loss/len(trainloader)}")

epoch=1/10 & loss=0.4554142733054883
epoch=2/10 & loss=0.27675073265012645
epoch=3/10 & loss=0.23229184301534314
epoch=4/10 & loss=0.1994135765148315
epoch=5/10 & loss=0.17493561454125242
epoch=6/10 & loss=0.1524400817873731
epoch=7/10 & loss=0.13252749319857499
epoch=8/10 & loss=0.11305503511447897
epoch=9/10 & loss=0.0985827124598168
epoch=10/10 & loss=0.08210132893563103


In [16]:
# Evaludate our CNN

correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in testloader:
        outputs = model.forward(images)
        _, predicted  = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"accuracy = {correct_labels / total_labels * 100}")

accuracy = 91.86999999999999
